# PS4 Analysis: S&P 500 Prices and Earnings

This notebook reproduces the analysis for the uploaded `MFE230E_PS4_data.csv` file.

It covers:

1. Stationarity tests for log prices and log earnings  
2. Stationarity test for the log price-earnings ratio  
3. VECM estimation  
4. Long-horizon regressions for horizons \(k=1,2,3,4,5\)  
5. Heteroskedasticity and serial-correlation diagnostics  
6. Comparison of OLS, White, Newey-West, and Hansen-Hodrick standard errors


In [1]:

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.vecm import VECM
from statsmodels.stats.diagnostic import het_white, acorr_ljungbox

# =========================
# Load data
# =========================
file_path = "MFE230E_PS4_data.csv"   # change path if needed
df = pd.read_csv(file_path)
df["Date"] = pd.to_datetime(df.iloc[:, 0])
df = df[["Date", "P", "E"]].copy()

# logs
df["p"] = np.log(df["P"])
df["e"] = np.log(df["E"])
df["pe"] = df["p"] - df["e"]

# =========================
# ADF helper
# =========================
def adf_table(series, name):
    rows = []
    for reg, label in [("c", "constant"), ("ct", "constant+trend"), ("n", "none")]:
        stat, pval, usedlag, nobs, crit, icbest = adfuller(series.dropna(), regression=reg, autolag="AIC")
        rows.append({
            "series": name,
            "spec": label,
            "adf_stat": stat,
            "p_value": pval,
            "lags": usedlag,
            "nobs": nobs
        })
    return pd.DataFrame(rows)

adf_results = pd.concat([
    adf_table(df["p"], "log price"),
    adf_table(df["e"], "log earnings"),
    adf_table(df["pe"], "log P/E"),
    adf_table(df["p"].diff().dropna(), "Δ log price"),
    adf_table(df["e"].diff().dropna(), "Δ log earnings"),
], ignore_index=True)

# =========================
# VECM
# =========================
# VAR lag selection in levels (up to 5)
var_order = VAR(df[["p", "e"]]).select_order(5)

# VECM with rank=1 and one lag of differences
vecm = VECM(df[["p", "e"]], k_ar_diff=1, coint_rank=1, deterministic="co").fit()

# =========================
# Long-horizon regressions
# Δp_{t+1}+...+Δp_{t+k} = p_{t+k} - p_t
# Δe_{t+1}+...+Δe_{t+k} = e_{t+k} - e_t
# =========================
for k in range(1, 6):
    df[f"y_p_{k}"] = df["p"].shift(-k) - df["p"]
    df[f"y_e_{k}"] = df["e"].shift(-k) - df["e"]

def hh_cov(results, nlags):
    """Hansen-Hodrick covariance with uniform weights."""
    X = np.asarray(results.model.exog)
    u = np.asarray(results.resid)
    Xu = X * u[:, None]
    S = Xu.T @ Xu
    for j in range(1, nlags + 1):
        Gamma = Xu[j:].T @ Xu[:-j]
        S += Gamma + Gamma.T
    XX_inv = np.linalg.inv(X.T @ X)
    return XX_inv @ S @ XX_inv

def run_lh_reg(dep, x, horizon):
    d = pd.DataFrame({"y": dep, "x": x}).dropna()
    X = sm.add_constant(d["x"])
    ols = sm.OLS(d["y"], X).fit()
    white = ols.get_robustcov_results(cov_type="HC0")
    nw = ols.get_robustcov_results(cov_type="HAC", maxlags=horizon - 1)

    hh_se = np.sqrt(np.diag(hh_cov(ols, horizon - 1)))

    white_test = het_white(ols.resid, X)   # LM stat, LM p, F stat, F p
    lb = acorr_ljungbox(ols.resid, lags=[max(1, horizon - 1)], return_df=True)

    return {
        "n": int(ols.nobs),
        "alpha": ols.params.iloc[0],
        "beta": ols.params.iloc[1],
        "R2": ols.rsquared,

        "se_OLS_alpha": ols.bse.iloc[0],
        "se_OLS_beta": ols.bse.iloc[1],

        "se_White_alpha": white.bse[0],
        "se_White_beta": white.bse[1],

        "se_NW_alpha": nw.bse[0],
        "se_NW_beta": nw.bse[1],

        "se_HH_alpha": hh_se[0],
        "se_HH_beta": hh_se[1],

        "t_OLS_beta": ols.tvalues.iloc[1],
        "t_White_beta": white.tvalues[1],
        "t_NW_beta": nw.tvalues[1],
        "t_HH_beta": ols.params.iloc[1] / hh_se[1],

        "White_pvalue": white_test[1],
        "LjungBox_pvalue": lb["lb_pvalue"].iloc[0],
    }

rows = []
for k in range(1, 6):
    out_p = run_lh_reg(df[f"y_p_{k}"], df["pe"], k)
    out_p["dep"] = f"Δp horizon {k}"
    rows.append(out_p)

    out_e = run_lh_reg(df[f"y_e_{k}"], df["pe"], k)
    out_e["dep"] = f"Δe horizon {k}"
    rows.append(out_e)

lh_results = pd.DataFrame(rows)

# =========================
# Save outputs
# =========================
adf_results.to_csv("adf_results.csv", index=False)
lh_results.to_csv("long_horizon_results.csv", index=False)

with open("vecm_summary.txt", "w", encoding="utf-8") as f:
    f.write(str(var_order.summary()))
    f.write("\n\n")
    f.write(str(vecm.summary()))

print("Done.")
print("Saved:")
print("- adf_results.csv")
print("- long_horizon_results.csv")
print("- vecm_summary.txt")


Done.
Saved:
- adf_results.csv
- long_horizon_results.csv
- vecm_summary.txt
